## Imports

and setup

In [ ]:
# import packages
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from tqdm import tqdm
from torchinfo import summary
import os
import matplotlib.pyplot as plt


import math
from pathlib import Path

# .py scripts imports
import sys
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root))
from utils.data_load import DataModule # used to load dataset and create dataloaders
from utils.checkpoint import ModelCheckpoint
from utils.logging import save_history
from utils.losses import (masked_l1_loss, # for 0.5 and 2.0 second 
                          masked_l1_grad_loss,
                          masked_huber_loss,
                          masked_multires_l1_loss, # for 3.0 and 5.0 second gaps

                          # evaluation metrics
                          masked_mae,
                          masked_rmse,
                          full_mae,
                          full_rmse,
                          psnr
                        )

In [ ]:
# choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# define checkpoints and history 
root_dir = Path("training_logs/unet")
# root_dir.mkdir(parents=True, exist_ok=True)

checkpoint_dir = root_dir / "model_weights"
history_dir = root_dir / "history_csv"

## Data Loading

To change?

In [ ]:
# load in dataset and create dataloaders using data_load.datamodule class
# using datamodule class to load data
dm = DataModule(
    repo_id="han2o/grant-ortsaem-processedV2", # hugging face repo id for dataset
    gap="0.5",  #  gap version [0.5, 2.0, 3.0, 5.0]
    input_key="masked_spectrogram",
    target_key="spectrogram",
    mask_key="mask",
    batch_size=8, # batch size for dataloaders
    num_workers=0,
    streaming=True,
    # maximum number of training/ validation/ test samples. Set to None to use the entire dataset.
    max_train_samples=128, 
    max_val_samples=32,
    max_test_samples=32, 
)

train_loader, val_loader, test_loader = dm.setup()

In [ ]:
# inspect one batch of data
batch = next(iter(train_loader))

print("keys:", batch.keys())
print("x shape:", batch["x"].shape)
print("y shape:", batch["y"].shape)
print("mask shape:", batch["mask"].shape)

if "gap_seconds" in batch:
    print("example gap:", batch["gap_seconds"][0])

In [ ]:
# visualise a sample spectrogram, mask, and target
x = batch["x"][0, 0].cpu().numpy()
y = batch["y"][0, 0].cpu().numpy()
m = batch["mask"][0, 0].cpu().numpy()

plt.figure(figsize=(25, 8))

plt.subplot(1, 3, 1)
plt.imshow(x, aspect="auto", origin="lower")
plt.title("Masked Spectrogram (x)", weight="bold")

plt.subplot(1, 3, 2)
plt.imshow(y, aspect="auto", origin="lower")
plt.title("Clean Spectrogram (y)", weight="bold")

plt.subplot(1, 3, 3)
plt.imshow(m, aspect="auto", origin="lower")
plt.title("Mask", weight="bold")

plt.tight_layout()
plt.show()

## U-Net 

### U-Net Parts
Forms the individual blocks in the U-Net archicture. The following code chunk consists of:


| Component | Purpose | Spatial Effect | Channel Effect |
|-----------|---------|----------------|----------------|
| `DoubleConv` | Feature extraction via two consecutive conv layers | Unchanged | in → out |
| `Down` | Encoder block (downsample + extract features) | H, W halved | Increases |
| `Up` | Decoder block (upsample + merge skip connection) | H, W doubled | Decreases |
| `OutConv` | Final 1×1 projection to output channels | Unchanged | → 1 (reconstruction) |


In [ ]:
# U-Net parts - from the unet repository

# =============================================================================
# U-Net Building Blocks
# 
# U-Net Architecture:
#   - Encoder: progressively downsamples spatial dimensions while increasing channels
#   - Decoder: progressively upsamples while decreasing channels
#   - Skip connections: concatenate encoder features to decoder
#
#   Input (1, 128, 256)  →  Encoder  →  Bottleneck  →  Decoder  →  Output (1, 128, 256)
#        ↓                                                ↑
#        └──────────── Skip Connections ──────────────────┘
# =============================================================================

class DoubleConv(nn.Module):
    """
    (convolution => [BN] => ReLU) * 2
    fundamental building block of U-Net, applies two sets of Conv-BatchNorm-ReLU

    """
   
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """
    Downscaling with maxpool then double conv
    Reduces spatial dimensions by a factor of 2 and increases channels from in_channels to out_channels
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """
    Upscaling then double conv
    Increases spatial dimensions by a factor of 2 and reduces channels from in_channels to out_channels
    
    If bilinear is True, uses nn.Upsample for upscaling and a DoubleConv with mid_channels = in_channels // 2 to reduce channels
    """

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
    
    # forward method applies upscaling, concatenates with the corresponding feature map from the encoder (x2), and applies double convolution
    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # if you have padding issues, see
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    """
    Final convolution layer to map to the desired number of output channels
    Uses a 1x1 convolution to reduce the number of channels to out_channels
    """
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

### U-Net Assembly

Uses the parts above to form a complete neural network. The following assembly is based off the U-Net repository. 


In [ ]:
""" Full assembly of the parts to form the complete network """

class UNet(nn.Module):
    """
    UNet architecture consisting of an encoder (Down blocks), a bottleneck, and a decoder (Up blocks) with skip connections.
    """

    def __init__(self, n_channels, n_classes, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        
        # 1 DoubleCOnv block for initial convolution 
        self.inc = (DoubleConv(n_channels, 64))
        # 3 down blocks , each reduces spatial dimensions by a factor of 2 and increases channels
        self.down1 = (Down(64, 128))
        self.down2 = (Down(128, 256))
        self.down3 = (Down(256, 512))
        # bottleneck
        factor = 2 if bilinear else 1
        # 1 down block for bottleneck, reduces spatial dimensions by a factor of 2 and increases channels
        self.down4 = (Down(512, 1024 // factor))

        # 4 up blocks, each increases spatial dimensions by a factor of 2 and reduces channels, with skip connections from the corresponding down blocks
        self.up1 = (Up(1024, 512 // factor, bilinear))
        self.up2 = (Up(512, 256 // factor, bilinear))
        self.up3 = (Up(256, 128 // factor, bilinear))
        self.up4 = (Up(128, 64, bilinear))
        self.outc = (OutConv(64, n_classes))


    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

    def use_checkpointing(self):
        self.inc = torch.utils.checkpoint(self.inc)
        self.down1 = torch.utils.checkpoint(self.down1)
        self.down2 = torch.utils.checkpoint(self.down2)
        self.down3 = torch.utils.checkpoint(self.down3)
        self.down4 = torch.utils.checkpoint(self.down4)
        self.up1 = torch.utils.checkpoint(self.up1)
        self.up2 = torch.utils.checkpoint(self.up2)
        self.up3 = torch.utils.checkpoint(self.up3)
        self.up4 = torch.utils.checkpoint(self.up4)
        self.outc = torch.utils.checkpoint(self.outc)

For the purposes of audio inpainting, the following modifications could be useful 